# Lab 03: Neural Network from Scratch

In this lab, you will how to build a L-layer neural network from scratch. We shall build a classifier that can detect cats. 

**Content**:
1. [Implementing an L-layered Neural Network](#Section-1.-Implementing-an-L-layered-Neural-Network)
    1. [Initialization](#1.1-Initialization)
    2. [Forward Propagation](#1.2-Forward-Propagation)
        1. [Forward Propagation (1 Layer)](#A.-Forward-Propagation-(1-Layer))
        2. [Forward Propagation through L Layers](#B.-Forward-Propagation-through-L-Layers)
    3. [Cost-function](#1.3-Cost-function)
    4. [Backward propagation](#1.4-Backward-propagation)
        1. [Backward Propagation (1 Layer)](#A.-Backward-Propagation-(1-Layer))
        2. [Backward Propagation through L Layers](#B.-Backward-Propagation-through-L-Layers) 
    5. [Update Parameters](#1.5-Update-Parameters)
3. [Section 2: Building a Cat Detector](#Section-2:-Building-a-Cat-Detector)
    1. [Loading the Dataset](#2.1-Loading-the-Dataset)
    2. [Cat detector with 2-Layer Neural Network](#2.2-Cat-detector-with-2-Layer-Neural-Network)
    3. [Cat detector with 4-Layer Neural Network](#2.3-Cat-detector-with-4-Layer-Neural-Network)


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import numpy as np
from utils import sigmoid, relu, sigmoid_backward, relu_backward
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

np.random.seed(1)


# Section 1. Implementing an L-layered Neural Network

In the this section, you will implement all the functions required to build an L-layered the network shown in the figure below. 

<img src="https://github.com/thkhoon/UCCD3074_Lab3/blob/master/imgs/deep_nn.png?raw=true" width=50%>

Here is the outline of the things we are going to do in this section.
1. Section 1.1: Initialize the parameters 
2. Section 1.2: Implement the forward propagation modules 
3. Section 1.3: Compute the loss.
3. Section 1.4: Implement the backward propagation module
4. Section 1.5: Update the parameters

---

### 1.1 Initialization

First, let's create the function to create and initialize the parameters for an L-layer neural network. 

<font color="blue">
    
#### Exercise 1

Implement the initialization function for an L-layer Neural Network. 
* Use small random initialization (`np.random.randn(<shape>)*0.01`) for the weight matrices $W^{[1]}$,  $W^{[2]}$, ...,  $W^{[L]}$.
* Use zero initialization (`np.zeros(<shape>)`)for the biases $b^{[1]}$,  $b^{[2]}$, ...,  $b^{[L]}$.

The input argument `layer_sizes` is a  tuple containing the size of each layer, i.e., ($n_x$, $n^{[1]}$, $n^{[2]}$, ..., $n^{[L-1]}$, $n_y$). For example, `(12288, 7, 1)` denotes a 2-layer neural network with the following layer sizes:
- $n_x$ = 12288  (each sample has 12288 features)
- $n^{[1]}$ = 7 (#unit in layer 1)
- $n^{[2]} = n_y$ = 1 (binary classification)

The output `parameters` is a dictionary `parameters` that stores the parameters of all layers:
```
parameters = {'W1': ...,
              'b1': ..., 
                 ...
              'WL': ...,
              'bL': ...
              }
```

Expected output:

```
Shape of W1: (7, 12288)
Shape of b1: (7, 1)
Shape of W2: (1, 7)
Shape of b2: (1, 1)

Initialization for layer 2:
W2:
 [[-8.53306509e-03  2.62677554e-02  1.44462274e-03  2.81582154e-02
  -4.48247037e-05  1.56583537e-02 -2.90084643e-03]]
b2:
 [[0.]]
```
</font>

In [ ]:
def init_params(layer_sizes):
    """
    Arguments:
    layer_sizes -- python array (list) containing the dimensions of each layer in our network
    
    Returns:
    parameters -- python dictionary containing your parameters "W1", "b1", ..., "WL", "bL":
                    Wl -- weight matrix of shape (layer_sizes[l], layer_sizes[l-1])
                    bl -- bias vector of shape (layer_sizes[l], 1)
    """
    
    np.random.seed(1)
    parameters = {}
    L = len(layer_sizes)            # number of layers in the network

    for l in range(1, L):
        
        ### START CODE HERE ### (≈ 2 lines of code)
        # ... 
        ### END CODE HERE ###
        
        assert(parameters['W' + str(l)].shape == (layer_sizes[l], layer_sizes[l-1]))
        assert(parameters['b' + str(l)].shape == (layer_sizes[l], 1))
        
    return parameters

Let's test your implementation

In [ ]:
# test case
layer_sizes = (12288, 7, 1)

# evaluate init_param
parameters = init_params(layer_sizes)

for key, param in parameters.items():
    print('Shape of ' + key + ":", param.shape)
    
print('\nInitialization for layer 2:')
print('W2:\n', parameters['W2'])
print('b2:\n', parameters['b2'])

---
### 1.2 Forward Propagation


### A. Forward Propagation (1 Layer)

Now, implement the `forward_one_layer` function.

<img src="https://github.com/thkhoon/UCCD3074_Lab3/blob/master/imgs/forward_one_layer.png?raw=true" width=50%>

The function `forward_one_layer` computes the value of $A^{[l]}$ through two phases:

#### Linear Phase

The linear phase computes the following equations: 

$$Z^{[l]} = W^{[l]}A^{[l-1]} +b^{[l]}\tag{1}$$

where $A^{[0]} = X$.
    
The phase will also generate the following cache value:
    
$$\mathrm{linear\_cache} = (A^{[l-1]}, W^{[l]}, b^{[l]})$$

#### Activation Phase

The activation phase subjects the weighted input $Z^{[l]}$ to a non-linear function. We implement two activation functions. In our network, we use relu activations for the hidden layers and the sigmoid activations for the output layer.

- **Sigmoid**: 
<br> The function implements the sigmoid function $\sigma(Z^{[l]}) = \frac{1}{1+e^{-Z^{[l]}}}$. It also caches $activation\_cache = Z^{[l]}$ for backpropagation later. We have provided you with the `sigmoid` function. You do not have to implement it yourself.

$$A, \mathrm{activation\_cache} = \mathrm{sigmoid}(Z)\tag{2a}$$
</font></center>

- **ReLU**: 
<br> The function implements the relu function $relu(Z) = \max(0, Z)$. It also caches $activation\_cache = Z^{[l]}$ for backpropagation later. We have provided you with the `relu` function. You do not have to implement it yourself.

$$A, \mathrm{activation\_cache} = \mathrm{relu}(Z)\tag{2b}$$


<font color="blue">

#### Exercise 2: 
    
Implement the linear activation function as described above and then verify your function.

In [ ]:
def forward_one_layer(A_prev, W, b, activation_function):
    """
    Implement the forward propagation for the LINEAR->ACTIVATION layer

    Arguments:
    A_prev -- activations from previous layer (or input data): (size of previous layer, number of examples)
    W -- weights matrix: numpy array of shape (size of current layer, size of previous layer)
    b -- bias vector, numpy array of shape (size of the current layer, 1)
    activation -- the activation to be used in this layer, stored as a text string: "sigmoid" or "relu"

    Returns:
    A -- the output of the activation function, also called the post-activation value 
    cache -- a python dictionary containing "linear_cache" and "activation_cache";
             stored for computing the backward pass efficiently
    """
    
    # Phase 1: Summation
    # Input: "A_prev, W, b"
    # Outputs: "Z, linear_cache"

    ### START CODE HERE ### (≈ 2 lines of code)
    # ... 
    ### END CODE HERE ###
    
    assert(Z.shape == (W.shape[0], A_prev.shape[1]))
    
    # Phase 2: Activation
    # Input: "Z"
    # Outputs: "A, activation_cache"
    
    ### START CODE HERE ### (≈ 1 line of code for each activation)
    if activation_function == "sigmoid":
        # ... 
    elif activation_function == "relu":
        # ... 
    else:
        print('unsupported activation', activation)
    ### END CODE HERE ###
        
    assert (A.shape == Z.shape)
    
    cache = (linear_cache, activation_cache)

    return A, cache

<font color="blue">
    
Verify `forward_one_layer` using the following test case. Consider layer 1 with $n^{[1]} = 3$ neurons, respectively. The test set contains m=4 samples randomly generated from a normal distribution. Each sample has input_dim $n_x = 2$. Check that the output tensor `A1` is a tensor of size (3, 4).
    
<img src="https://github.com/thkhoon/UCCD3074_Lab3/blob/master/imgs/forward_one_layer_testcase.png?raw=true" width=40%>

Expected output:
```
A1:
 [[0.50484916 0.5000596  0.50178867 0.49957471]
 [0.49377443 0.50653091 0.50044359 0.50135947]
 [0.49536232 0.51020543 0.50254536 0.50167027]]
```


In [ ]:
#settings
layer_sizes = (2, 3)
m = 4

# generate test case
parameters = init_params(layer_sizes)
A0 = np.random.randn(layer_sizes[0], m)

# evaluate forward_one_layer
A1, cache1 = forward_one_layer(A0, parameters['W1'], parameters['b1'], activation_function = "sigmoid")

print('A1:\n', A1)

assert (cache1[0][0] == A0).all()
assert (cache1[0][1] == parameters['W1']).all()
assert (cache1[0][2] == parameters['b1']).all()

### B. Forward Propagation through L Layers

In this section, you shall implement the forward propagation algorithm through an $L$-layered neural network. The model is parameterized by `parameter`. Given an input signal `X`, forward propagation computes the predicted output $\hat{Y}$. 

<img src="https://github.com/thkhoon/UCCD3074_Lab3/blob/master/imgs/forward_prop.png?raw=true" width=70%>

<font color="blue">
    
#### Exercise 3

Complete the function `forward` which implements the forward propagation of the above model. 
* Use **relu** activation for all hidden layers, i.e., layer 1, 2, ..., $L-1$
* Use **sigmoid** activation for the output layer, i.e., layer $L$
* In each iteration, store all the `cache` value into the dictionary `caches`. Name the key for each layer as `cache1, cache2, ..., cache[L]`
* Use a `for` loop to perform foward propagation for the hidden layers by calling the function `forward_one_layer` that you have just implemented.

In [ ]:
def forward(X, parameters, activation_functions):
    """
    Implement forward propagation for the [LINEAR->RELU]*(L-1)->LINEAR->SIGMOID computation
    
    Arguments:
    X:  ndarray of size (nx, m) input matrix
    parameters:  a dictionary of the network parameters, e.g., 
                 for 2-layered NN {'W1': ..., 'b1':..., 'W2':..., 'b2': ...}
    activation_functions: the list of activation functsions used in each layer (including input layer). 
                The first value is always None. 
                E.g., if we use relu for 1st layer and sigmoid for 2nd layer: [None, 'relu', 'sigmoid']
    
    Returns:
    AL -- last post-activation value
    caches -- list of caches containing:
              every cache of linear_relu_forward() (there are L-1 of them, indexed from 0 to L-2)
              the cache of linear_sigmoid_forward() (there is one, indexed L-1)
    """

    caches = {}
    A = X
    L = len(parameters) // 2                  # number of layers in the neural network
    
    # Implement Forward propagation
    # Input: 'A_prev'
    # Output: 'A' and 'caches'
    #         'linear_cache' added into the dictionary 'cache' with the key 
    #                 "cache" + str(l)". For example, for layer 1, the name
    #                 for the cache is "cache1"
    for l in range(1,L+1):
        A_prev = A 
        ### START CODE HERE ### (≈ 2 lines of code)
        # ... 
        ### END CODE HERE ###
        
    assert(A.shape[1] == X.shape[1])
            
    return A, caches

<font color="blue">
    
Verify `forward` using the following test case. Consider a 2-layer NN with layer sizes = `(2, 3, 3)`, respectively and activation functions `(None, relu, sigmoid)`. The test set contains m=4 samples randomly generated from a normal distribution. Each sample has input_dim $n_x = 2$. Check that the output tensor $A^{[2]}$ is a tensor of size (3, 4).

<img src="https://github.com/thkhoon/UCCD3074_Lab3/blob/master/imgs/forward_testcase.png?raw=true" width=50%>
    
Expected output:
```
AL
[[0.5        0.50001196 0.5        0.5       ]
 [0.5        0.49992126 0.5        0.5       ]
 [0.5        0.50005219 0.5        0.5       ]]
Length of caches list = 2
```
</font>

In [ ]:
# settings
layer_sizes = (2, 3, 3)                   
activation_functions = (None, "relu", "sigmoid")
m = 4                                         

# create random test set
parameters = init_params(layer_sizes)         # initialize parameters
X = np.random.randn(layer_sizes[0], m)        # create input

# evaluate forward_propagation
AL, caches = forward(X, parameters, activation_functions) # forward propagation

print("AL\n" + str(AL))
print("Length of caches list = " + str(len(caches)))

---
### 1.3 Cost function

The forward propagation is used for inference where the model predicts the output layer $\hat{Y}$. Now, you implement the cost, to check how well your model is actually learning. We shall use the binary **cross entropy function**. 

$$J(\hat{Y}, Y) = -\frac{1}{m} \sum\limits_{i = 1}^{m} (Y^{(i)}\log \hat{Y}^{(i)} + (1-Y^{(i)})\log (1- \hat{Y}^{(i)}) \tag{3}$$


<font color="blue">

**Exercise 4**

Compute the cross-entropy cost $J$, using the formula above: 
Expected output:
```
cost = 0.6431332160756019
```

</font>

In [ ]:
def compute_cost(AL, Y):
    """
    Implement the cost function defined by equation (7).

    Arguments:
    AL -- probability vector corresponding to your label predictions, shape (1, number of examples)
    Y -- true "label" vector (for example: containing 0 if non-cat, 1 if cat), shape (1, number of examples)

    Returns:
    cost -- cross-entropy cost
    """
    
    m = Y.shape[1]

    # Compute loss from aL and y.
    ### START CODE HERE ### (≈ 1 lines of code)
    # ... 
    ### END CODE HERE ###
    
    cost = np.squeeze(cost)      # To make sure your cost's shape is what we expect (e.g. this turns [[17]] into 17).
    assert(cost.shape == ())
    
    return cost

Test your implementation. Here, we are using a 2-layered neural network to verify your code.

In [ ]:
# create test case
layer_sizes = (12288, 7, 1)                   
activation_functions = (None, 'relu', 'sigmoid')
nx, n1, ny = layer_sizes                      # no of units in each layer
m = 1                                         # no of samples

parameters = init_params(layer_sizes)         # initialize parameters
X = np.random.randn(nx, m)                    # create input
Y = np.random.randn(ny, m)                    # create output

predicted, caches = forward(X, parameters, activation_functions) # forward propagation

In [ ]:
# evaluate compute_cost
cost = compute_cost(predicted, Y)
print("cost = " + str(cost))

---
### 1.4 Backward propagation


### A. Backward Propagation (1 Layer)

Next, you will create the `backward_one_layer` function performs the backpropagation through 1 FC layer. 

<img src="https://github.com/thkhoon/UCCD3074_Lab3/blob/master/imgs/backprop_one_layer.png?raw=true" width=50%>

The backpropagation commences in two phase:

#### Activation

The activation function $g(Z^{[l]})$ can be a ReLU or sigmoid function. Given the upstream gradient $dA^{[l]}$ and $activation\_cache$, the task is to compute $dZ^{[l]}$. To help you implement `backward_one_layer`, we provided two backward functions to do just that:
- `sigmoid_backward`: Implements the backward propagation for a *sigmoid* unit. You can call it as follows:

$$dZ = \mathrm{sigmoid\_backward} (dA, \mathrm{activation\_cache})\tag{4a}$$

- `relu_backward`: Implements the backward propagation for a *relu* unit. You can call it as follows:

$$dZ = \mathrm{relu\_backward} (dA, \mathrm{activation\_cache})\tag{4b}$$


#### Linear

The summation function is given by $Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}$. During backpropagation. Given $dZ^{[l]}$, you want to get $dW^{[l]}$, $db^{[l]}$ and $dA^{[l-1]}$. 

Here are the formulas you need:

$$ dW^{[l]} = \frac{\partial \mathcal{L} }{\partial W^{[l]}} = \frac{1}{m} dZ^{[l]} A^{[l-1] T} \tag{5}$$

$$ db^{[l]} = \frac{\partial \mathcal{L} }{\partial b^{[l]}} = \frac{1}{m} \sum_{i = 1}^{m} dZ^{[l](i)}\tag{6}$$

$$ dA^{[l-1]} = W^TdZ \tag{7}$$

Note that the values $A^{[l-1]}$, $W^{[l]}$ and $b^{[l]}$ is from the `cache` output of the `forward_one_layer` function.

<font color="blue">

#### Exercise 5
    
Implement the `backward_one_layer` function to perform backpropagation through 1 layer of fully connected layer.
   
Expected answer:

```
----- sigmoid -----
dA0
[[ 0.00137895  0.00137857  0.00137838  0.00137869]
 [ 0.00063364  0.00063401  0.00063415  0.00063392]
 [-0.00707273 -0.00707391 -0.00707425 -0.00707372]]
dW1
[[ 0.06582484 -0.08153407 -0.06352779]
 [ 0.06580578 -0.08154494 -0.06353438]]
db1
[[0.24999547]
 [0.24997396]]

----- relu -----
dA0
[[ 0.01624345  0.00551377  0.01624345  0.00551377]
 [-0.00611756  0.00253651 -0.00611756  0.00253651]
 [-0.00528172 -0.0282971  -0.00528172 -0.0282971 ]]
dW1
[[ 0.2633184  -0.32612608 -0.25410211]
 [-0.25264432 -0.61104877 -0.49443742]]
db1
[[1. ]
 [0.5]]
```
</font>

In [ ]:
def backward_one_layer(dA, cache, activation_function):
    """
    Implement the backward propagation for the LINEAR->ACTIVATION layer.
    
    Arguments:
    dA -- post-activation gradient for current layer l 
    cache -- tuple of values (linear_cache, activation_cache) we store for computing backward propagation efficiently
    activation -- the activation to be used in this layer, stored as a text string: "sigmoid" or "relu"
    
    Returns:
    dA_prev -- Gradient of the cost with respect to the activation (of the previous layer l-1), same shape as A_prev
    dW -- Gradient of the cost with respect to W (current layer l), same shape as W
    db -- Gradient of the cost with respect to b (current layer l), same shape as b
    """
    linear_cache, activation_cache = cache    
    m = dA.shape[1]
    
    # back propagation through the activation function
    
    ### START CODE HERE ### (≈ 2 lines of code)
    if activation_function == "relu":
        # ... 
    elif activation_function == "sigmoid":
        # ... 
    ### END CODE HERE ###

    assert (dZ.shape == dA.shape)
        
    # back propagation through the linear function

    A_prev, W, b = linear_cache

    ### START CODE HERE ### (≈3 lines of code)
    # ... 
    ### END CODE HERE ###

    assert (dA_prev.shape == A_prev.shape)
    assert (dW.shape == W.shape)
    assert (db.shape == b.shape)
        
    return dA_prev, dW, db

<font color="blue">
    
Verify `backward_one_layer` using the following test case. Consider layer 1 with $n^{[1]}=3$ neurons, respectively. The test set contains m=4 samples randomly generated from a normal distribution. Each sample has input_dim $n_x = 2$. Check that the gradiant $dA1$ is a tensor of size (3, 4), $dW^{[1]}$ of size (3,2) and $db^{[1]}$ of size (3,1).


<img src="https://github.com/thkhoon/UCCD3074_Lab3/blob/master/imgs//backprop_one_layer_testcase.png?raw=true" width="35%">

In [ ]:
#settings
layer_sizes = (2, 3)
m = 4

# generate test case
parameters = init_params(layer_sizes)
A0 = np.random.randn(layer_sizes[0], m)

A1, cache1 = forward_one_layer(A0, parameters['W1'], parameters['b1'], activation_function = "sigmoid")
dA1 = np.ones_like(A1)  # randomly generate dA1 for testing purpose

<font color="blue">

Expected answer when the layer uses sigmoid activation.
```
dA0
[[ 0.00490359  0.00490328  0.00490385  0.00490394]
 [-0.0099646  -0.0099628  -0.00996549 -0.00996557]]
dW1
[[ 0.0658191  -0.08153984]
 [ 0.06582091 -0.08152353]
 [ 0.06583969 -0.08148495]]
db1
[[0.24999328]
 [0.24997914]
 [0.24996627]]
```

In [ ]:
# evaluate backward_one_layer for sigmoid activation function
dA0, dW1, db1 = backward_one_layer(dA1, cache1, activation_function = "sigmoid")
print ("----- sigmoid -----")
print ("dA0\n"+ str(dA0))
print ("dW1\n" + str(dW1))
print ("db1\n" + str(db1) + "\n")

<font color="blue">

Expected answer when the layer uses relu activation.

```
----- relu -----
dA0
[[ 0.01624345  0.01961581  0.01961581  0.00337236]
 [-0.00611756 -0.03986264 -0.03986264 -0.03374507]]
dW1
[[ 0.32566099 -0.23011249]
 [-0.17288455 -0.69165307]
 [-0.17288455 -0.69165307]]
db1
[[0.75]
 [0.75]
 [0.75]]
```

In [ ]:
# evaluate backward_one_layer for relu activation function
dA0, dW1, db1 = backward_one_layer(dA1, cache1, activation_function = "relu")

print ("----- relu -----")
print ("dA0\n"+ str(dA0))
print ("dW1\n" + str(dW1))
print ("db1\n" + str(db1) + "\n")

### B. Backward Propagation through L Layers

Now you will implement the `backward` function to implement the backpropagation for the whole network. The backpropagation algorithm computes the **gradient** of the loss w.r.t. the model parameters, i.e., $dW^{[1]}$, $db^{[1]}$, ..., $dW^{[L]}$, $db^{[L]}$. The gradients are then used to update the model parameters. 

<img src="https://github.com/thkhoon/UCCD3074_Lab3/blob/master/imgs/backprob.png?raw=true" width="70%">

The implementation of the backpropagation function commences in two steps. In the `backward` function, you will backpropate the gradients starting from the loss function, through the output layer followed by all the hidden layers in the backward order. 

* **Loss Function**
    
    First, we backpropagate through the binary cross entropy loss function $J(\hat{Y}, Y)$ and compute the gradient of the output signal $dA^{[L]}$. To do so, use the following formula:

$$dA^{[L]} = \frac{1 - Y}{1 - A^{[L]}} - \frac{Y}{A^{[L]}}\tag{6}$$

* **FC Layers**

    The next step is to compute the gradients associated with the output layer, namely $dW^{[l]}$ and $db^{[l]}$ for all hidden layers $l = 1...L$ . In your implementation, you should call the function:
    
$$\mathrm{backward\_one\_layer}(\mathrm{dA\_prev}, \mathrm{cache[l]}, \mathrm{activation\_functions[l]})\tag{7}$$

The output should be stored in the **dictionary** `grads` which contains the following keys:

```
grad['dW1'] = ... 
grad['db1'] = ...
...
```

<font color="blue">
    
#### Exercise 6

Implement the `backward` function.

In [ ]:
def backward(AL, Y, activation_functions, caches):
    """
    Implement the backward propagation for the [LINEAR->RELU] * (L-1) -> LINEAR -> SIGMOID group
    
    Arguments:
    AL -- probability vector, output of the forward propagation (forward_propagation())
    Y -- true "label" vector (containing 0 if non-cat, 1 if cat)
    caches -- list of caches containing:
                every cache of forward_one_layer() with "relu" (it's caches[l], for l in range(L-1) i.e l = 0...L-2)
                the cache of forward_one_layer() with "sigmoid" (it's caches[L-1])
    activation_functions -- list of activaiton functions for each layer
   
    Returns:
    grads -- A dictionary with the gradients
             grads["dA" + str(l)] = ... 
             grads["dW" + str(l)] = ...
             grads["db" + str(l)] = ... 
    """
    grads = {}
    L = len(caches) # the number of layers
    m = AL.shape[1]
    Y = Y.reshape(AL.shape) # after this line, Y is the same shape as AL
    
    # Backpropagation for loss function:  
    ### START CODE HERE ### (1 line of code)
    # ...
    ### END CODE HERE ###
            
    # Backpropagation for layer l to L-1
    dA_prev = dAL
    for l in reversed(range(1, L+1)):
        ### START CODE HERE (approx. 4 lines) ####
        # ... Get cache for layer l ...
        # ... Compute dA_prev, dW, db ...
        # ... save dW into "grads" ...
        # ... save db into "grads" ...
        ### END CODE HERE ###

    return grads

<font color="blue">
    
Verify `backward` using the following test case. Consider a 2-layer NN with layer sizes = `(2, 3, 3)`, respectively and activation functions `(None, relu, sigmoid)`. The test set contains m=4 samples randomly generated from a normal distribution. Each sample has input_dim $n_x = 2$.

<img src="https://github.com/thkhoon/UCCD3074_Lab3/blob/master/imgs/backprob_testcase.png?raw=true" width=50%>

```
dW1:
 [[ 0.0004993   0.00318705]
 [-0.00039636 -0.00253001]
 [ 0.00026833  0.00171278]]
db1:
 [[-0.00289569]
 [ 0.00229871]
 [-0.0015562 ]]
dW2:
 [[-0.00049152 -0.00158996 -0.0029798 ]
 [ 0.00049146  0.00158975  0.0029794 ]
 [ 0.00049159  0.00159017  0.00298018]]
db2:
 [[-2.49997010e-01]
 [-1.96838683e-05]
 [-2.49986953e-01]]
```

In [ ]:
layer_sizes = (2, 3, 3)                   
activation_functions = (None, "relu", "sigmoid")
m = 4                                         

# create random test set
parameters = init_params(layer_sizes)         # initialize parameters
X = np.random.randn(layer_sizes[0], m)        # create input
Y = np.random.randint(0, 2, (layer_sizes[-1], m))

# evaluate forward_propagation
AL, caches = forward(X, parameters, activation_functions) # forward propagation
cost       = compute_cost(AL, Y)

In [ ]:
grads = backward(AL, Y, activation_functions, caches)
print('dW1:\n', grads['dW1'])
print('db1:\n', grads['db1'])
print('dW2:\n', grads['dW2'])
print('db2:\n', grads['db2'])

---
### 1.5 Update Parameters

In this section you will update the parameters of the model, using gradient descent: 

$$ W^{[l]} = W^{[l]} - \alpha \text{ } dW^{[l]} \quad \forall l\tag{8}$$

$$ b^{[l]} = b^{[l]} - \alpha \text{ } db^{[l]} \quad \forall l\tag{9}$$

where $\alpha$ is the learning rate. Store all $dW^{[l]}$ and $db^{[l]}$ into the `parameters` dictionary. 

<font color="blue">
    
#### Exercise 7

Implement `update_parameters()` to update your parameters using gradient descent. Update on every $W^{[l]}$ and $b^{[l]}$ for $l = 1, 2, ..., L$. 

Expected output:
```
W1
[[ 0.01612938 -0.00619839]
 [-0.00511619 -0.01057431]
 [ 0.00882296 -0.0230619 ]]
b1
[[ 0.00037008]
 [-0.0002299 ]
 [-0.00012791]]
W2
[[ 0.01720529 -0.00779569  0.00310053]
 [-0.00296781  0.0143999  -0.02098964]
 [-0.00375027 -0.0036569   0.01088085]]
b2
[[-4.44036059e-06]
 [-2.49963245e-02]
 [-2.50012987e-02]]
```
</font>

In [ ]:
def update_parameters(parameters, grads, learning_rate):
    """
    Update parameters using gradient descent
    
    Arguments:
    parameters -- python dictionary containing your parameters 
    grads -- python dictionary containing your gradients, output of backward_propagation
    
    Returns:
    parameters -- python dictionary containing your updated parameters 
                  parameters["W" + str(l)] = ... 
                  parameters["b" + str(l)] = ...
    """
    
    L = len(parameters) // 2 # number of layers in the neural network
    # Update rule for each parameter. Use a for loop.
    ### START CODE HERE ### (≈ 3 lines of code)
    # ...
    ### END CODE HERE ###
    return parameters

Test your implementation here:

In [ ]:
np.random.seed(3)
activation_functions = (None, 'relu', 'sigmoid')
layer_sizes = (2, 3, 3)                 
m = 4                                       

# Creating test cases
X   = np.random.randn(layer_sizes[0], m)  
Y   = np.random.randint(0, 2, (layer_sizes[-1], m))  

# perform forward propagation
parameters = init_params(layer_sizes)       
AL, caches = forward(X, parameters, activation_functions)    

# perform backpropagation
grads = backward(AL, Y, activation_functions, caches)

In [ ]:
lr = 0.1
parameters = update_parameters(parameters, grads, lr)  # update parameters

print ("W1\n"+ str(parameters["W1"]))
print ("b1\n"+ str(parameters["b1"]))
print ("W2\n"+ str(parameters["W2"]))
print ("b2\n"+ str(parameters["b2"]))

---
## Section 2: Building a Cat Detector

In the following, your task is to use a L-layer Neural Network to build a cat detector.

<img src="https://github.com/thkhoon/UCCD3074_Lab3/blob/master/imgs/cat_detector.png?raw=true" width="90%">

### 2.1 Loading the Dataset

We will use the cat dataset that we used in our practical dataset as last week. There are 209 training samples and 50 test samples. Each sample is a color image with `(64, 64)`. The labels can be positive (`y=1`) and negative (`y=0`).

In [ ]:
from utils import load_dataset

X_train_ori, Y_train, X_test_ori, Y_test, classes = load_dataset()

m_train = len(X_train_ori)  # number of training samples
m_test = len(X_test_ori)    # number of test samples

Flatten each sample to get the matrix of size `(nx, m) = (12288, m)`

In [ ]:
X_train_flatten = X_train_ori.reshape(m_train, -1).T
X_test_flatten =  X_test_ori.reshape(m_test, -1).T

Preprocess the data so that the pixel values lie in the range [0, 1]

In [ ]:
X_train = X_train_flatten/255.
X_test = X_test_flatten/255.

---
### 2.2 Cat detector with 2-Layer Neural Network

Now that you are familiar with the dataset, it is time to build a deep neural network to distinguish cat images from non-cat images. You will build a L-layer deep neural network where L denotes the depth of the neural network. 
<br><br>


### A. Training

<font color='blue'>

#### Exercise 8

Build a **2-layer** neural network with `layer_sizes = (12288, 7, 1)` and `activation_functions=(None, 'relu', 'sigmoid')`. To do this, you need to implement the following deep learning methodology:
1. Initialize parameters
2. Loop for num_iterations:
    1. Forward propagation
    2. Compute cost function
    3. Backward propagation
    4. Update parameters (using parameters, and grads from backprop) 

Expected output:

```
Cost after iteration 1: 0.697252
Cost after iteration 201: 0.507265
Cost after iteration 401: 0.441614
Cost after iteration 601: 0.342711
Cost after iteration 801: 0.225049
Cost after iteration 1001: 0.136700
Cost after iteration 1201: 0.223034
Cost after iteration 1401: 0.077355
Cost after iteration 1601: 0.056353
Cost after iteration 1801: 0.041995
Cost after iteration 2000: 0.033591
```
</font>

In [ ]:
def build_model(X, Y, layer_sizes, activation_functions, lr = 0.01, num_iters = 3000, verbose=True):
    """
    Implements a L-layer neural network: [LINEAR->RELU]*(L-1)->LINEAR->SIGMOID.
    
    Arguments:
    X -- data, numpy array of shape (number of examples, num_px * num_px * 3)
    Y -- true "label" vector (containing 0 if cat, 1 if non-cat), of shape (1, number of examples)
    layers_dims -- list containing the input size and each layer size, of length (number of layers + 1).
    learning_rate -- learning rate of the gradient descent update rule
    num_iterations -- number of iterations of the optimization loop
    print_cost -- if True, it prints the cost every 100 steps
    
    Returns:
    parameters -- parameters learnt by the model. They can then be used to predict.
    """

    np.random.seed(1)
    hist = []                         # keep track of cost
    
    ### START CODE HERE ###
    # Parameters initialization.
    # ...

    
    # Loop (gradient descent)
    for i in range(0, num_iters):

        # Forward propagation
        # ...
        
        # Compute cost
        # ...
        
        # Backpropagation
        # ...
 
        # Update network parameters
        # ...
                
        # Print the cost every 100 training example
        if verbose and i % 200 == 0 or i == num_iters-1:
            hist.append(cost)
            print ("Cost after iteration %i: %f" %(i+1, cost))
            
    ### END CODE ###
    
    return parameters, hist

In [ ]:
# Train a 2 layer model 
layer_sizes = (12288, 7, 1)
activation_functions = (None, 'relu', 'sigmoid')

parameters, hist = build_model(X_train, Y_train, layer_sizes, activation_functions, lr=0.02, num_iters = 2000)

Now, let's plot the training history. You can see that the cost are decreasing 

In [ ]:
# plot the cost
plt.plot(hist)
plt.ylabel('cost')
plt.xlabel('iterations (per tens)')
plt.title("Learning rate =" + str(lr))
plt.show()

### B. Evaluation

From previously, you have trained a cat detector. The parameters are saved into `parameters`. Next, the function `evaluate` uses the model to predict the label (cat / no cat) on images and then evaluate their accuracy. 

Expected output:
```
Training accuracy = 100.00%
Testing accuracy = 70.00%
```

In [ ]:
def evaluate(X, Y_actual, parameters, activation_functions):
    """
    This function is used to predict the results of a  L-layer neural network.
    
    Arguments:
    X -- data set of examples you would like to label
    parameters -- parameters of the trained model
    
    Returns:
    p -- predictions for the given dataset X
    """
    
    m = X.shape[1]
    
    ### START CODE HERE ###
    # Forward propagation
    # ...

    # compute accuracy
    # ...
    ### END CODE HERE ###
        
    return Y_pred, acc

In [ ]:
pred_train, acc_train = evaluate (X_train, Y_train, parameters, activation_functions)
print(f"Training accuracy = {acc_train*100:.2f}%")
      
pred_test, acc_test = evaluate (X_test, Y_test, parameters, activation_functions)
print(f"Testing accuracy = {acc_test*100:.2f}%")

For the 2-layer model, the model performs very well on the training set on which it is trained (100%) but achieves a much lower performance (70%) on the test set which it has not seen before. 

---
### 2.3 Cat detector with 4-Layer Neural Network

Would you do better with a deeper neural network? In the following, let's try a deeper neural network with 5 layers, and see if a 5-layer model performs any better than a 2-layer neural network. Remember that we achieve a testing accuracy of around 70% for the 2-layer neural network.


### A. Training

<font color="blue">
    
#### Exercise 8

Train a cat detector using a **5-layer** neural network with `layer_sizes = (12288, 20, 7, 5, 1)`, and an activation function of `(None, 'relu', 'relu', 'relu', 'sigmoid')`

```
Cost after iteration 1: 0.693149
Cost after iteration 201: 0.651095
Cost after iteration 401: 0.645075
Cost after iteration 601: 0.644150
Cost after iteration 801: 0.644002
Cost after iteration 1001: 0.643978
Cost after iteration 1201: 0.643975
Cost after iteration 1401: 0.643974
Cost after iteration 1601: 0.643974
Cost after iteration 1801: 0.643974
Cost after iteration 2000: 0.643974
```

In [ ]:
### START CODE (~3 lines) ###
#  ...
### END CODE HERE ###

### B. Evaluation

<font color="blue">
    
#### Exercise 9
    
Compute the training and testing accuracy for the 5-layer NN. 

Expected answer:
```
Training accuracy = 65.55%
Testing accuracy = 34.00%
```

In [ ]:
### START CODE (~2 lines) ###
#  ...
### END CODE ###

print(f"Training accuracy = {acc_train*100:.2f}%")
print(f"Testing accuracy = {acc_test*100:.2f}%")

The 5-layer neural network seems to suffer from both **lower training** accuracy of 65.55% and **lower testing** accuracy of 34% compared to the 2-layer neural network which delivered training accuracy 100% and test accuracy 68%. Why does a 2-layer neural network performs better than a 5-layer neural network?  
* Although both training and testing accuracies are lower, this is **not** due to overfitting (The 5-layered is a more powerful architecture than the 2-layered version).  
* It turns out that training a **deep** network is not as simple as adding layers to the architecture. The optimizer is having difficulty training the network.

How to solve this problem? Over the years, we have learnt a great deal about how to train a deep layer. We can get a better result if we use the 
1. **Xavier** initialization scheme 
2. **Batch normalization** 

We shall learn how to do just that in another lab.